In [ ]:
# =========================================
# IMPORT LIBRARIES
# =========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print("Libraries imported successfully!")

In [ ]:
# =========================================
# COLUMN NAMES
# =========================================

column_names = [
    'sepal_length',
    'sepal_width',
    'petal_length',
    'petal_width',
    'species'
]

# =========================================
# LOAD DATASET
# =========================================

df = pd.read_csv(
    "Iris.csv",
    names=column_names
)

print("FIRST 5 ROWS")
print(df.head())

print("\nDATASET SHAPE")
print(df.shape)


In [ ]:
# =========================================
# (a) DATA CLEANING
# =========================================

# Replace ? with NaN
df.replace("?", np.nan, inplace=True)

# Convert numeric columns to numeric
for col in df.columns[:-1]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Check missing values
print("\nMISSING VALUES")
print(df.isnull().sum())

# Remove missing values
df_cleaned = df.dropna()

# Remove negative values (from numeric columns only)
numeric_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]

df_cleaned = df_cleaned[
    (df_cleaned[numeric_cols] >= 0).all(axis=1)
]

print("\nSHAPE AFTER CLEANING")
print(df_cleaned.shape)


In [ ]:
# =========================================
# (b) ERROR CORRECTING
# OUTLIER DETECTION & REMOVAL (IQR)
# =========================================

def remove_outliers_iqr(df, columns):

    for col in columns:

        Q1 = df[col].quantile(0.25)

        Q3 = df[col].quantile(0.75)

        IQR = Q3 - Q1

        lower = Q1 - 1.5 * IQR

        upper = Q3 + 1.5 * IQR

        df = df[
            (df[col] >= lower) &
            (df[col] <= upper)
        ]

    return df

df_no_outliers = remove_outliers_iqr(
    df_cleaned.copy(),
    numeric_cols
)

print("\nSHAPE AFTER OUTLIER REMOVAL")
print(df_no_outliers.shape)


In [ ]:
# =========================================
# (c) DATA TRANSFORMATION
# FEATURE SCALING
# =========================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    df_no_outliers[numeric_cols]
)

# Features and target
X = X_scaled

y = df_no_outliers['species']

print("\nDATA TRANSFORMATION COMPLETED")

# Show transformed data
df_scaled = pd.DataFrame(X_scaled, columns=numeric_cols)
print(df_scaled.head())


In [ ]:
# =========================================
# TRAIN TEST SPLIT
# =========================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

In [ ]:
# =========================================
# (d) LOGISTIC REGRESSION MODEL
# =========================================

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)

accuracy_lr = accuracy_score(y_test, log_pred)

print("Logistic Regression Accuracy:", round(accuracy_lr * 100, 2), "%")


In [ ]:
# =========================================
# NAIVE BAYES MODEL
# =========================================
nb_model = GaussianNB()

nb_model.fit(X_train, y_train)

nb_pred = nb_model.predict(X_test)
accuracy_nb = accuracy_score(y_test, nb_pred)

print("Naive Bayes Accuracy:", round(accuracy_nb * 100, 2), "%")

In [ ]:
# =========================================
# ACCURACY COMPARISON
# =========================================

print("\n================================")
print("MODEL ACCURACY")
print("================================")

print("\nLogistic Regression Accuracy:")
print(accuracy_lr)

print("\nNaive Bayes Accuracy:")
print(accuracy_nb)


In [ ]:
# =========================================
# CLASSIFICATION REPORTS
# =========================================

print("\n================================")
print("\nLOGISTIC REGRESSION REPORT")

print(classification_report(
    y_test,
    log_pred
))

print("\n================================")
print("NAIVE BAYES REPORT")
print("================================")

print(classification_report(
    y_test,
    nb_pred
))

In [ ]:
# =========================================
# CONFUSION MATRIX VISUALIZATION
# LOGISTIC REGRESSION
# =========================================

sns.heatmap(
    confusion_matrix(y_test, log_pred),
    annot=True,
    fmt='d'
)

plt.title("Logistic Regression Confusion Matrix")

plt.show()

# =========================================
# CONFUSION MATRIX VISUALIZATION
# NAIVE BAYES
# =========================================

sns.heatmap(
    confusion_matrix(y_test, nb_pred),
    annot=True,
    fmt='d'
)

plt.title("Naive Bayes Confusion Matrix")

plt.show()

# =========================================
# FINAL COMPARISON
# =========================================

print("\n================================")
print("FINAL COMPARISON")
print("================================")

if accuracy_lr > accuracy_nb:

    print("\nLogistic Regression Performs Better")

elif accuracy_nb > accuracy_lr:

    print("\nNaive Bayes Performs Better")

else:

    print("\nBoth Models have Equal Accuracy")